# Лабораторная работа 5. Диффузионные модели (Stable Diffusion)

В этой лабораторной работе мы дообучим модель **Stable Diffusion v1.5** с помощью метода **DreamBooth** и **LoRA**, чтобы она научилась генерировать изображения с конкретным человеком (мной).

**Цели:**
1. Настроить окружение и загрузить базовую модель.
2. Сгенерировать изображения класса (Prior Preservation) для предотвращения деградации модели.
3. Обучить LoRA-адаптер на 10-15 фотографиях пользователя.
4. Сгенерировать изображения в разных стилях (киберпанк, ведьмак и т.д.).
5. Оценить качество генераций с помощью метрик (Face Similarity, CLIP Score, BRISQUE).


## §1. Dev — Подготовка окружения
Клонирование или обновление репозитория (Git Pull).


In [ ]:
import os
import sys
from pathlib import Path

if "COLAB_GPU" in os.environ or "COLAB_RELEASE_TAG" in os.environ:
    from google.colab import userdata  # type: ignore
    github_token = userdata.get('GITHUB_TOKEN')
    repo_url = f"https://{github_token}@github.com/Ma-XD/ITMO-CV.git"
    
    if not Path("/content/ITMO-CV").exists():
        !git clone {repo_url} /content/ITMO-CV
    else:
        %cd /content/ITMO-CV
        !git pull
    
    sys.path.insert(0, "/content/ITMO-CV/lab5-DIF")
    %cd /content/ITMO-CV/lab5-DIF
else:
    LAB_DIR = Path(".").resolve()
    if LAB_DIR.name != "lab5-DIF":
        %cd lab5-DIF
    if str(LAB_DIR) not in sys.path:
        sys.path.insert(0, str(LAB_DIR))


## §2. Установка зависимостей
Монтирование Google Drive (в Colab) и установка библиотек.


In [ ]:
from env_config import is_colab

if is_colab:
    from google.colab import drive  # type: ignore
    # drive.mount идемпотентен: если уже примонтирован, он просто напишет 'Drive already mounted'
    drive.mount('/content/drive')
    
    # pip install идемпотентен: если пакеты уже установлены, он напишет 'Requirement already satisfied'
    !pip install -r requirements.txt


## §3. Импорты и проверка конфигурации
Загрузка путей из `config.py` и проверка доступности GPU.


In [ ]:
import sys
from pathlib import Path
import torch
import matplotlib.pyplot as plt
from PIL import Image

# Добавляем папку лабы в sys.path для импорта конфигов
lab_dir = Path("lab5-DIF").resolve() if not is_colab else Path("/content/ITMO-CV/lab5-DIF")
if str(lab_dir) not in sys.path:
    sys.path.insert(0, str(lab_dir))

from env_config import print_env, get_device
from config import (
    INSTANCE_DIR, CLASS_DIR, CHECKPOINT_DIR, FIGURE_DIR,
    MODEL_ID, INSTANCE_PROMPT, CLASS_PROMPT, RESOLUTION
)

print_env()
device = get_device()

print(f"\nПапка с фото пользователя (Instance): {INSTANCE_DIR}")
print(f"Папка с фото класса (Class): {CLASS_DIR}")
print(f"Чекпоинты сохраняются в: {CHECKPOINT_DIR}")


## §4. Загрузка базовой модели
Загружаем оригинальную модель Stable Diffusion v1.5 (идемпотентно).


In [ ]:
from diffusers import StableDiffusionPipeline
import torch

try:
    if pipeline is not None:
        print("✅ Базовая модель уже находится в памяти.")
except NameError:
    print("Загрузка базовой модели SD 1.5...")
    pipeline = StableDiffusionPipeline.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.float16 if device.type == "cuda" else torch.float32,
        safety_checker=None  # Отключаем safety checker
    )
    pipeline = pipeline.to(device)
    if device.type == "cuda":
        pipeline.enable_attention_slicing()
    print("✅ Базовая модель успешно загружена!")


### Smoke-test базовой модели
Генерируем тестовое изображение до дообучения, чтобы убедиться в работоспособности.


In [ ]:
# Smoke-test: генерация тестового изображения
prompt = "a photo of a cat"
generator = torch.Generator(device=device).manual_seed(42)

print(f"Генерируем: '{prompt}'...")
test_image = pipeline(prompt, generator=generator, num_inference_steps=30).images[0]

# Отрисовка результата
plt.figure(figsize=(6, 6))
plt.imshow(test_image)
plt.axis('off')
plt.title("Smoke-test: Базовая модель SD 1.5")
plt.show()


## §5. Генерация изображений класса (Prior Preservation)
Создаем 150 изображений обычных людей, чтобы модель не забыла их после дообучения на лице пользователя. Ячейка идемпотентна.


In [ ]:
import hashlib
from tqdm.auto import tqdm

num_class_images = 150
existing_images = list(CLASS_DIR.glob("*.jpg")) + list(CLASS_DIR.glob("*.png"))

if len(existing_images) >= num_class_images:
    print(f"✅ Изображения класса уже существуют ({len(existing_images)} шт.). Пропускаем генерацию.")
else:
    images_to_generate = num_class_images - len(existing_images)
    print(f"Сбор изображений класса. Нужно сгенерировать {images_to_generate} шт.")
    
    # Отключаем градиенты для ускорения генерации
    with torch.no_grad():
        for i in tqdm(range(images_to_generate), desc="Генерация class images"):
            # Генерируем изображение
            image = pipeline(CLASS_PROMPT, num_inference_steps=30).images[0]
            
            # Сохраняем с уникальным именем (хэш)
            hash_name = hashlib.sha1(image.tobytes()).hexdigest()[:10]
            image_path = CLASS_DIR / f"class_{hash_name}.jpg"
            image.save(image_path)
            
    print("✅ Генерация изображений класса завершена.")


## §6. Обучение DreamBooth LoRA
Напишем класс датасета и сам цикл обучения. Ячейка обучения идемпотентна.

In [ ]:
import random
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

class DreamBoothDataset(Dataset):
    def __init__(self, instance_dir, class_dir, tokenizer, size=512):
        self.instance_paths = [p for p in Path(instance_dir).iterdir() if p.is_file() and p.suffix.lower() in ['.jpg', '.jpeg', '.png']]
        self.class_paths = [p for p in Path(class_dir).iterdir() if p.is_file() and p.suffix.lower() in ['.jpg', '.jpeg', '.png']]
        
        if not self.instance_paths:
            raise ValueError(f"❌ Папка {instance_dir} пуста! Добавьте свои фото.")
            
        self.tokenizer = tokenizer
        self.size = size
        self.transform = transforms.Compose([
            transforms.Resize((size, size)),
            transforms.ToTensor(),
            transforms.Normalize([0.5], [0.5])
        ])

    def __len__(self):
        # Искусственно увеличиваем длину эпохи
        return len(self.instance_paths) * 100

    def __getitem__(self, index):
        inst_path = self.instance_paths[index % len(self.instance_paths)]
        class_path = random.choice(self.class_paths)

        inst_img = Image.open(inst_path).convert("RGB")
        class_img = Image.open(class_path).convert("RGB")

        inst_prompt_ids = self.tokenizer(
            INSTANCE_PROMPT, truncation=True, padding="max_length", 
            max_length=self.tokenizer.model_max_length, return_tensors="pt"
        ).input_ids[0]

        class_prompt_ids = self.tokenizer(
            CLASS_PROMPT, truncation=True, padding="max_length", 
            max_length=self.tokenizer.model_max_length, return_tensors="pt"
        ).input_ids[0]

        return {
            "instance_images": self.transform(inst_img),
            "instance_prompt_ids": inst_prompt_ids,
            "class_images": self.transform(class_img),
            "class_prompt_ids": class_prompt_ids
        }

In [ ]:
import torch.nn.functional as F
from peft import LoraConfig, get_peft_model
from diffusers import DDPMScheduler
from tqdm.auto import tqdm

lora_path = CHECKPOINT_DIR / "adapter_model.safetensors"

if lora_path.exists():
    print("✅ LoRA уже обучена и сохранена. Пропускаем обучение.")
else:
    print("Подготовка к обучению...")
    unet = pipeline.unet
    text_encoder = pipeline.text_encoder
    vae = pipeline.vae
    tokenizer = pipeline.tokenizer
    noise_scheduler = DDPMScheduler.from_pretrained(MODEL_ID, subfolder="scheduler")

    # Замораживаем базовые веса
    vae.requires_grad_(False)
    text_encoder.requires_grad_(False)
    unet.requires_grad_(False)

    # Настраиваем LoRA (только для UNet attention)
    lora_config = LoraConfig(
        r=8, lora_alpha=8, 
        target_modules=["to_q", "to_k", "to_v", "to_out.0"]
    )
    unet = get_peft_model(unet, lora_config)
    unet.to(device)

    # Датасет и оптимизатор
    dataset = DreamBoothDataset(INSTANCE_DIR, CLASS_DIR, tokenizer, RESOLUTION)
    dataloader = DataLoader(dataset, batch_size=1, shuffle=True)
    optimizer = torch.optim.AdamW(unet.parameters(), lr=1e-4)

    num_steps = 1000
    progress_bar = tqdm(total=num_steps, desc="Обучение LoRA")
    step = 0

    unet.train()
    while step < num_steps:
        for batch in dataloader:
            if step >= num_steps:
                break

            # Конкатенируем instance и class для Prior Preservation
            images = torch.cat([batch["instance_images"], batch["class_images"]]).to(device)
            prompt_ids = torch.cat([batch["instance_prompt_ids"], batch["class_prompt_ids"]]).to(device)

            # Переводим картинки в латентное пространство
            latents = vae.encode(images.to(dtype=vae.dtype)).latent_dist.sample()
            latents = latents * vae.config.scaling_factor

            # Добавляем шум
            noise = torch.randn_like(latents)
            bsz = latents.shape[0]
            timesteps = torch.randint(0, noise_scheduler.config.num_train_timesteps, (bsz,), device=device).long()
            noisy_latents = noise_scheduler.add_noise(latents, noise, timesteps)

            # Получаем эмбеддинги текста
            encoder_hidden_states = text_encoder(prompt_ids)[0]

            # Предсказываем шум
            model_pred = unet(noisy_latents, timesteps, encoder_hidden_states).sample

            # Считаем Loss (Prior Preservation: вес 1.0)
            model_pred_inst, model_pred_class = model_pred.chunk(2)
            noise_inst, noise_class = noise.chunk(2)

            loss_inst = F.mse_loss(model_pred_inst, noise_inst, reduction="mean")
            loss_class = F.mse_loss(model_pred_class, noise_class, reduction="mean")
            loss = loss_inst + loss_class

            loss.backward()
            optimizer.step()
            optimizer.zero_grad()

            progress_bar.update(1)
            progress_bar.set_postfix({"loss": loss.item()})
            step += 1

    # Сохраняем веса
    unet.save_pretrained(CHECKPOINT_DIR)
    print(f"✅ Обучение завершено! Веса LoRA сохранены в {CHECKPOINT_DIR}")